# Notebook 1: continuous qHTS potencies

The qHTS features are the only input in this project that has ever moved blind Spearman,
and they were built from the wrong export. `aid1851_concise.csv` carries Active/Inactive
calls only. The full assay data table carries `Fit_LogAC50`, the fitted dose-response
potency, for **43,110 compound-isoform pairs**, and it carries SMILES inline so there is
no CID crawl.

| panel | fitted AC50 values | our own labels |
| :--- | ---: | ---: |
| CYP1A2 | 9,849 | 1,203 |
| CYP2C19 | 9,563 | none, usable as an auxiliary task |
| CYP3A4 | 9,384 | 2,016 |
| CYP2C9 | 8,329 | 1,091 |
| CYP2D6 | 5,984 | 1,273 |

CYP1A2 matters most. STATUS.md records that CYP1A2 had no public potency data at all,
which is why it was left out of the Veith work. That was wrong: it has the most of any
panel here, and it is one of the two endpoints where we are weakest.

`Fit_LogAC50` is log10 of the AC50 in molar, verified against the `Potency` column in
micromolar at a median ratio of exactly 1, so **pAC50 = -Fit_LogAC50**. That lands at
2.05 to 9.10, against the challenge's own 1.9 to 8.0.

Runtime: GPU, T4. Nothing to upload.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU')

In [ ]:
# Challenge data, folds, holdout mask and frozen embeddings, loaded from Drive.
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/mini3_d3108f9c6e.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
os.makedirs('/content/results', exist_ok=True)
OUTDIR = '/content/results'
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
print(sorted(os.listdir('/content/cache')))

In [ ]:
!pip -q install tabpfn==2.2.1 rdkit lightgbm

In [ ]:
import json, time, numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold, KFold

CACHE = Path('/content/cache'); OUT = Path(OUTDIR)
ISOFORMS = ['CYP1A2', 'CYP2C9', 'CYP2D6', 'CYP3A4']
N_FOLDS = 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

meta = pd.read_csv(CACHE / 'meta_train.csv.gz')
mete = pd.read_csv(CACHE / 'meta_test.csv.gz')
hold = np.load(CACHE / 'holdout_mask.npy')
G = np.load(CACHE / 'scaffold_groups.npy', allow_pickle=True)
EMB = np.load(CACHE / 'emb_medium_inhib.npy').astype(np.float32)
EMB_TE = np.load(CACHE / 'emb_medium_test.npy').astype(np.float32)
QH = np.load(CACHE / 'qhts_tr.npy')          # the binary-call features, for comparison
QH_TE = np.load(CACHE / 'qhts_te.npy')
AUX = np.load(CACHE / 'aux_train.npy'); AUX_TE = np.load(CACHE / 'aux_test.npy')
SM_TR = meta['SMILES'].to_numpy(); SM_TE = mete['SMILES'].to_numpy()

# The shipped blend, per isoform, as the number every arm below is judged against.
SHIPPED = {'CYP1A2': 0.5791, 'CYP2C9': 0.7076, 'CYP2D6': 0.4263, 'CYP3A4': 0.8094}

def dev_index(iso):
    y = meta[f'{iso}_pIC50_direct_inhibition'].to_numpy(float)
    obs = np.isfinite(y)
    return y, np.flatnonzero(obs & ~hold), np.flatnonzero(obs & hold)

RESULTS = []
def record(**kw):
    RESULTS.append(kw)
    json.dump(RESULTS, open(OUT / 'results.json', 'w'), indent=2)

print('device', DEVICE, '| dev sizes', {i: len(dev_index(i)[1]) for i in ISOFORMS})

## Fetch and parse the assay data table

One request, about 23 MB. `skiprows=[1]` drops PubChem's RESULT_TYPE row, which otherwise
turns every numeric column into strings.

In [ ]:
URL = ('https://pubchem.ncbi.nlm.nih.gov/assay/pcget.cgi?query=download'
       '&record_type=datatable&actvty=all&response_type=save&aid=1851')
!wget -q -O /content/aid1851_full.csv "{URL}"

raw = pd.read_csv('/content/aid1851_full.csv', low_memory=False, skiprows=[1])
raw['pac50'] = -pd.to_numeric(raw['Fit_LogAC50'], errors='coerce')
raw['smiles'] = raw['PUBCHEM_EXT_DATASOURCE_SMILES']
PANEL = {'p450-cyp1a2': 'CYP1A2', 'p450-cyp2c9': 'CYP2C9', 'p450-cyp2c19': 'CYP2C19',
         'p450-cyp2d6': 'CYP2D6', 'p450-cyp3a4': 'CYP3A4'}
raw['panel'] = raw['Panel Name'].map(PANEL)
raw = raw.dropna(subset=['panel', 'smiles'])
print('rows', len(raw), '| unique compounds', raw.smiles.nunique())
print(raw[raw.pac50.notna()].groupby('panel').size().to_string())

## Drop anything the blind set can see

Canonicalised overlap against both the 750 blind compounds and our own 4,905 training
compounds. Dropping the training overlap is not about leakage into the leaderboard; it
keeps the attached feature honest for our own cross-validation, since a public model that
has already seen a training compound would hand it back an answer it could not give for
the blind set.

In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
RDLogger.DisableLog('rdApp.*')

def canon(s):
    m = Chem.MolFromSmiles(s) if isinstance(s, str) else None
    return Chem.MolToSmiles(m) if m is not None else None

ours = {c for c in map(canon, np.concatenate([SM_TR, SM_TE])) if c}
uniq = pd.DataFrame({'smiles': raw.smiles.unique()})
uniq['canon'] = [canon(s) for s in uniq.smiles]
uniq = uniq.dropna(subset=['canon'])
uniq['overlap'] = uniq.canon.isin(ours)
print(f'public compounds {len(uniq)}, overlapping a challenge compound {int(uniq.overlap.sum())}')
keep = set(uniq.loc[~uniq.overlap, 'smiles'])
pub = raw[raw.smiles.isin(keep) & raw.pac50.notna()][['smiles', 'panel', 'pac50']]
pub = pub.groupby(['smiles', 'panel'], as_index=False).pac50.median()
print('usable public measurements', len(pub))
print(pub.groupby('panel').size().to_string())

In [ ]:
names = [d[0] for d in Descriptors._descList]
calc = MoleculeDescriptors.MolecularDescriptorCalculator(names)
gen = AllChem.GetMorganGenerator(radius=2, fpSize=2048)

def featurise(smiles):
    X = np.zeros((len(smiles), 2048 + len(names)), np.float32)
    for i, s in enumerate(smiles):
        m = Chem.MolFromSmiles(s) if isinstance(s, str) else None
        if m is None:
            continue
        X[i, :2048] = gen.GetFingerprintAsNumPy(m)
        X[i, 2048:] = np.nan_to_num(np.asarray(calc.CalcDescriptors(m), np.float32),
                                    nan=0.0, posinf=0.0, neginf=0.0)
    return X

pub_smiles = pub.smiles.unique()
t0 = time.time()
PX = featurise(pub_smiles)
TRX, TEX = featurise(SM_TR), featurise(SM_TE)
pos = {s: i for i, s in enumerate(pub_smiles)}
print(f'featurised {PX.shape} public, {TRX.shape} train, {TEX.shape} test '
      f'({time.time()-t0:.0f}s)')

## Train one potency model per panel

Five continuous columns attached to every challenge compound, the same shape as the
binary features but carrying potency instead of a call. The self out-of-fold Spearman is
printed as a sanity bound: a feature cannot carry more signal than its own model has.
For reference, the Veith-derived models we tried previously reached 0.54, 0.47, 0.46 and
0.65 on their own held-out data and were useless as features.

In [ ]:
import lightgbm as lgb
PANELS = ['CYP1A2', 'CYP2C9', 'CYP2C19', 'CYP2D6', 'CYP3A4']
PARAMS = dict(objective='regression', learning_rate=0.03, num_leaves=63,
              min_data_in_leaf=20, feature_fraction=0.3, bagging_fraction=0.8,
              bagging_freq=1, lambda_l2=1.0, verbosity=-1, num_threads=4)

CONT_TR = np.full((len(SM_TR), len(PANELS)), np.nan, np.float32)
CONT_TE = np.full((len(SM_TE), len(PANELS)), np.nan, np.float32)
for j, p in enumerate(PANELS):
    sub = pub[pub.panel == p]
    idx = np.array([pos[s] for s in sub.smiles]); y = sub.pac50.to_numpy(float)
    oof = np.zeros(len(idx))
    for tr, va in KFold(5, shuffle=True, random_state=0).split(idx):
        m = lgb.train(PARAMS, lgb.Dataset(PX[idx[tr]], y[tr]), 800)
        oof[va] = m.predict(PX[idx[va]])
    m = lgb.train(PARAMS, lgb.Dataset(PX[idx], y), 800)
    CONT_TR[:, j] = m.predict(TRX); CONT_TE[:, j] = m.predict(TEX)
    r = float(spearmanr(y, oof).statistic)
    record(stage='public_model', panel=p, n=len(idx), self_oof_spearman=r)
    print(f'  {p}: n={len(idx)} self OOF Spearman {r:.4f}', flush=True)

np.save(OUT / 'qhts_cont_tr.npy', CONT_TR); np.save(OUT / 'qhts_cont_te.npy', CONT_TE)
print('\nhow well each continuous column ranks our own labels, against our own model:')
for iso in ISOFORMS:
    y, dev, _ = dev_index(iso)
    best = max(((abs(spearmanr(y[dev], CONT_TR[dev, j]).statistic), PANELS[j])
                for j in range(len(PANELS))))
    print(f'  {iso}: best column {best[1]} {best[0]:.4f}  (ours {SHIPPED[iso]:.4f})')

## Does TabPFN rank better with them

Three arms, raw features and no PCA, which is the configuration that gained +0.0125 in the
last notebook. `binary` is the control: it reproduces what we already ship, so if it drifts
far from 0.5766 / 0.7158 / 0.4388 / 0.8062 the comparison is not measuring what it claims.

In [ ]:
from tabpfn import TabPFNRegressor
KW = dict(device=DEVICE, random_state=42, ignore_pretraining_limits=True)

ARMS = {'binary':   (np.hstack([EMB, QH, AUX]),            np.hstack([EMB_TE, QH_TE, AUX_TE])),
        'continuous': (np.hstack([EMB, CONT_TR, AUX]),     np.hstack([EMB_TE, CONT_TE, AUX_TE])),
        'both':     (np.hstack([EMB, QH, CONT_TR, AUX]),   np.hstack([EMB_TE, QH_TE, CONT_TE, AUX_TE]))}

for name, (Xtr, Xte) in ARMS.items():
    Xtr = np.nan_to_num(Xtr.astype(np.float32)); Xte = np.nan_to_num(Xte.astype(np.float32))
    print(f'\n{name}  {Xtr.shape[1]} columns', flush=True)
    for iso in ISOFORMS:
        y, dev, ho = dev_index(iso)
        t0 = time.time()
        oof = np.full(len(dev), np.nan)
        for tr, va in GroupKFold(N_FOLDS).split(dev, dev, G[dev]):
            m = TabPFNRegressor(**KW).fit(Xtr[dev[tr]], y[dev[tr]])
            oof[va] = m.predict(Xtr[dev[va]])
        m = TabPFNRegressor(**KW).fit(Xtr[dev], y[dev]); hp = m.predict(Xtr[ho])
        allr = np.concatenate([dev, ho])
        m2 = TabPFNRegressor(**KW).fit(Xtr[allr], y[allr]); tp = m2.predict(Xte)
        np.savez_compressed(OUT / f'cont_{name}_{iso}.npz',
                            oof=oof, dev=dev, hold=hp, ho=ho, test=tp)
        d = float(spearmanr(y[dev], oof).statistic)
        h = float(spearmanr(y[ho], hp).statistic)
        record(stage='arm', arm=name, iso=iso, dev_spearman=d, holdout_spearman=h,
               shipped=SHIPPED[iso], seconds=round(time.time() - t0, 1))
        print(f'  {iso}: dev {d:.4f} | holdout {h:.4f}  ({time.time()-t0:.0f}s)', flush=True)

In [ ]:
r = pd.DataFrame([x for x in RESULTS if x.get('stage') == 'arm'])
tab = r.pivot(index='arm', columns='iso', values='dev_spearman')
tab['MACRO'] = tab.mean(axis=1)
tab.loc['control reference'] = [0.5766, 0.7158, 0.4388, 0.8062,
                                np.mean([0.5766, 0.7158, 0.4388, 0.8062])]
display(tab.round(4))
print('control drift:')
print((tab.loc['binary'] - tab.loc['control reference']).round(4).to_string())

import shutil
from google.colab import files
shutil.make_archive('/content/notebook1_results', 'zip', OUT)
files.download('/content/notebook1_results.zip')